# SemRel Dataset — Preprocessing Pipeline
**COS 760 Group Project**  
This notebook preprocesses the SemRel dataset for **Afrikaans**.
It loads the data, performs cleaning, and saves cleaned splits for model notebooks to consume.

## 1. Install & Import Dependencies

In [1]:
!pip install datasets transformers pandas numpy matplotlib seaborn langdetect -q

In [2]:
# Language code for SemRel / SemEval-2024 Task 1
LANGUAGES = ['afr']

LANGUAGE_NAMES = {
    'afr': 'Afrikaans',
}

SPLITS = ['train', 'dev', 'test']

# Output directory for cleaned data
OUTPUT_DIR = './cleaned_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Language: Afrikaans (afr)')
print(f'Output directory: {OUTPUT_DIR}')


NameError: name 'os' is not defined

In [ ]:
raw_data = {}

for lang in LANGUAGES:
    raw_data[lang] = {}
    for split in SPLITS:
        try:
            dataset = load_dataset(
                'SemRel/SemRel2024',
                lang,
                split=split,
                trust_remote_code=True
            )
            raw_data[lang][split] = dataset.to_pandas()
            print(f'Loaded Afrikaans {split}: {len(raw_data[lang][split])} rows')
        except Exception as e:
            print(f'Could not load afr {split}: {e}')

print('\nDataset loading complete.')


In [ ]:
# Inspect schema
if 'afr' in raw_data and 'train' in raw_data['afr']:
    df_sample = raw_data['afr']['train']
    print('Schema for Afrikaans train:')
    print(df_sample.dtypes)
    print()
    display(df_sample.head(5))
else:
    print('Afrikaans train data not available.')


# Dataset size summary
print('Dataset size summary (Afrikaans):')
print(f'{"Split":<10} {"Rows"}')
print('-' * 20)
for split in SPLITS:
    n = len(raw_data['afr'][split]) if 'afr' in raw_data and split in raw_data['afr'] else 'N/A'
    print(f'{split:<10} {n}')


In [ ]:
if 'afr' in raw_data and 'train' in raw_data['afr']:
    scores = raw_data['afr']['train']['score']
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(scores, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_title('Afrikaans — train score distribution')
    ax.set_xlabel('Relatedness score')
    ax.set_ylabel('Count')
    ax.axvline(scores.mean(), color='coral', linestyle='--', label=f'Mean: {scores.mean():.2f}')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/score_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Score distribution plot saved.')


# Descriptive statistics


In [ ]:
if 'afr' in raw_data and 'train' in raw_data['afr']:
    scores = raw_data['afr']['train']['score']
    print(f'Afrikaans:')
    print(f'  Min: {scores.min():.4f}  Max: {scores.max():.4f}  Mean: {scores.mean():.4f}  Std: {scores.std():.4f}')
    
# Inspect one language to understand the schema
sample_lang = LANGUAGES[0]
sample_split = 'train'

if sample_lang in raw_data and sample_split in raw_data[sample_lang]:
    df_sample = raw_data[sample_lang][sample_split]
    print(f'Schema for {LANGUAGE_NAMES[sample_lang]} {sample_split}:')
    print(df_sample.dtypes)
    print()
    display(df_sample.head(5))
else:
    print('Sample data not available.')

In [ ]:
# Dataset size summary
print('Dataset size summary:')
print(f'{"Language":<15} {"Train":<10} {"Dev":<10} {"Test":<10}')
print('-' * 45)
for lang in LANGUAGES:
    sizes = []
    for split in SPLITS:
        if lang in raw_data and split in raw_data[lang]:
            sizes.append(str(len(raw_data[lang][split])))
        else:
            sizes.append('N/A')
    print(f'{LANGUAGE_NAMES[lang]:<15} {sizes[0]:<10} {sizes[1]:<10} {sizes[2]:<10}')

In [ ]:
cleaned_data = {}

print('Preprocessing Afrikaans splits:')
print('-' * 45)

for split in SPLITS:
    if 'afr' in raw_data and split in raw_data['afr']:
        try:
            cleaned_data.setdefault('afr', {})
            cleaned_data['afr'][split] = preprocess_dataframe(
                raw_data['afr'][split], 'afr', split
            )
        except Exception as e:
            print(f'  Error preprocessing afr {split}: {e}')

print('\nPreprocessing complete.')


In [ ]:
if 'afr' in cleaned_data and 'train' in cleaned_data['afr']:
    df = cleaned_data['afr']['train']
    lengths_s1 = df['sentence1'].apply(lambda x: len(x.split()))
    lengths_s2 = df['sentence2'].apply(lambda x: len(x.split()))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(lengths_s1, bins=30, alpha=0.6, label='Sentence 1', color='steelblue')
    ax.hist(lengths_s2, bins=30, alpha=0.6, label='Sentence 2', color='coral')
    ax.set_title('Afrikaans — word count distribution (cleaned train)')
    ax.set_xlabel('Word count')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/sentence_lengths.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Sentence 1 avg: {lengths_s1.mean():.1f} words | Sentence 2 avg: {lengths_s2.mean():.1f} words')


In [ ]:
for split in SPLITS:
    if 'afr' in cleaned_data and split in cleaned_data['afr']:
        df = cleaned_data['afr'][split]
        filename = f'{OUTPUT_DIR}/afr_{split}.csv'
        df.to_csv(filename, index=False)
        print(f'Saved: {filename} ({len(df)} rows)')


In [ ]:
def load_split(lang: str, split: str, data_dir: str = './cleaned_data') -> pd.DataFrame:
    """
    Load a single language/split CSV.
    Usage:
        train_afr = load_split('afr', 'train')
        dev_afr   = load_split('afr', 'dev')
    """
    path = os.path.join(data_dir, f'{lang}_{split}.csv')
    if not os.path.exists(path):
        raise FileNotFoundError(f'File not found: {path}. Run preprocessing notebook first.')
    return pd.read_csv(path)


def load_all_splits(lang: str = 'afr', data_dir: str = './cleaned_data'):
    """
    Load train, dev, and test splits for Afrikaans.
    Returns: (train_df, dev_df, test_df)
    Usage:
        train, dev, test = load_all_splits()
    """
    return (
        load_split(lang, 'train', data_dir),
        load_split(lang, 'dev',   data_dir),
        load_split(lang, 'test',  data_dir),
    )


# Quick verification
print('Testing loader...')
try:
    sample = load_split('afr', 'train')
    print(f'Loaded {len(sample)} rows for Afrikaans train.')
    display(sample.head(3))
except FileNotFoundError as e:
    print(e)


In [ ]:
print('='*45)
print('PREPROCESSING COMPLETE — FINAL SUMMARY')
print('='*45)
print(f'{"Split":<10} {"Rows"}')
print('-'*20)

for split in SPLITS:
    if 'afr' in cleaned_data and split in cleaned_data['afr']:
        n = len(cleaned_data['afr'][split])
        print(f'{split:<10} {n}')

print()
print('Cleaned CSVs saved to:', OUTPUT_DIR)
print()
print('Load in your model notebook with:')
print('  train, dev, test = load_all_splits()')


## 7. Run Preprocessing

In [ ]:
cleaned_data = {}

print('Preprocessing all language/split combinations:')
print('-' * 55)

for lang in LANGUAGES:
    cleaned_data[lang] = {}
    for split in SPLITS:
        if lang in raw_data and split in raw_data[lang]:
            try:
                cleaned_data[lang][split] = preprocess_dataframe(
                    raw_data[lang][split], lang, split
                )
            except Exception as e:
                print(f'  Error preprocessing {lang} {split}: {e}')

print('\nPreprocessing complete.')

## 8. Sentence Length Analysis

In [ ]:
# Word count distribution per language (train split)
fig, axes = plt.subplots(1, len(LANGUAGES), figsize=(15, 4))
if len(LANGUAGES) == 1:
    axes = [axes]

for i, lang in enumerate(LANGUAGES):
    if lang in cleaned_data and 'train' in cleaned_data[lang]:
        df = cleaned_data[lang]['train']
        lengths_s1 = df['sentence1'].apply(lambda x: len(x.split()))
        lengths_s2 = df['sentence2'].apply(lambda x: len(x.split()))
        axes[i].hist(lengths_s1, bins=30, alpha=0.6, label='Sentence 1', color='steelblue')
        axes[i].hist(lengths_s2, bins=30, alpha=0.6, label='Sentence 2', color='coral')
        axes[i].set_title(f'{LANGUAGE_NAMES[lang]} — word counts')
        axes[i].set_xlabel('Word count')
        axes[i].set_ylabel('Frequency')
        axes[i].legend(fontsize=9)
        print(f'{LANGUAGE_NAMES[lang]} — Sentence 1 avg: {lengths_s1.mean():.1f} words | Sentence 2 avg: {lengths_s2.mean():.1f} words')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sentence_lengths.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Cleaned Data
Saves per-language CSV files and one combined CSV for all models to load from.

In [ ]:
all_frames = []

for lang in LANGUAGES:
    for split in SPLITS:
        if lang in cleaned_data and split in cleaned_data[lang]:
            df = cleaned_data[lang][split]

            # Per-language per-split file
            filename = f'{OUTPUT_DIR}/{lang}_{split}.csv'
            df.to_csv(filename, index=False)
            print(f'Saved: {filename} ({len(df)} rows)')

            all_frames.append(df)

# Combined file — all languages and splits
if all_frames:
    combined = pd.concat(all_frames, ignore_index=True)
    combined.to_csv(f'{OUTPUT_DIR}/all_languages_combined.csv', index=False)
    print(f'\nSaved combined file: {OUTPUT_DIR}/all_languages_combined.csv ({len(combined)} rows total)')

## 10. Helper Loaders for Each Model
Each person can use these functions in their own notebook to load the cleaned data.

In [ ]:
def load_split(lang: str, split: str, data_dir: str = './cleaned_data') -> pd.DataFrame:
    """
    Load a single language/split CSV.
    Usage:
        train_afr = load_split('afr', 'train')
        dev_afr   = load_split('afr', 'dev')
    """
    path = os.path.join(data_dir, f'{lang}_{split}.csv')
    if not os.path.exists(path):
        raise FileNotFoundError(f'File not found: {path}. Run preprocessing notebook first.')
    return pd.read_csv(path)


def load_all_splits(lang: str = 'afr', data_dir: str = './cleaned_data'):
    """
    Load train, dev, and test splits for Afrikaans.
    Returns: (train_df, dev_df, test_df)
    Usage:
        train, dev, test = load_all_splits()
    """
    return (
        load_split(lang, 'train', data_dir),
        load_split(lang, 'dev',   data_dir),
        load_split(lang, 'test',  data_dir),
    )


# Quick verification
print('Testing loader...')
try:
    sample = load_split('afr', 'train')
    print(f'Loaded {len(sample)} rows for Afrikaans train.')
    display(sample.head(3))
except FileNotFoundError as e:
    print(e)


## 11. Final Summary

In [ ]:
print('='*45)
print('PREPROCESSING COMPLETE — FINAL SUMMARY')
print('='*45)
print(f'{"Split":<10} {"Rows"}')
print('-'*20)

for split in SPLITS:
    if 'afr' in cleaned_data and split in cleaned_data['afr']:
        n = len(cleaned_data['afr'][split])
        print(f'{split:<10} {n}')

print()
print('Cleaned CSVs saved to:', OUTPUT_DIR)
print()
print('Load in your model notebook with:')
print('  train, dev, test = load_all_splits()')
